# Análisis de resultados – Práctica 3 (Datos Masivos II)
PageRank, HITS y PageRank Sensible a Tópico sobre **cit-HepPh**, **Spotify** y **Wikipedia** (muestra y, si terminó, completa).

Antes de abrir este cuaderno hay que haber corrido, en este orden: `src/ingest/load_hepph.py`, `src/ingest/load_spotify.py`,
`src/ingest/parse_wiki.py` y `build_wiki_graph.py` (muestra), y `src/algos/run_all.py`. Este cuaderno solo **lee** los resultados
de `results/<dataset>/` y no usa Spark.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED, RESULTS = ROOT / "data" / "processed", ROOT / "results"
FIGS = ROOT / "report" / "figuras"; FIGS.mkdir(parents=True, exist_ok=True)
TABLES = RESULTS / "tablas"; TABLES.mkdir(parents=True, exist_ok=True)

# Datasets disponibles (los que ya tienen resultados) y la columna con el nombre legible del nodo
LABEL = {"hepph": "paper", "spotify": "name", "wiki_sample": "title", "wiki_full": "title"}
DATASETS = [d for d in LABEL if (RESULTS / d / "ranks.parquet").exists()]
print("Datasets con resultados:", DATASETS)
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 60)

## 1. Carga y tamaños

In [ ]:
def load(name):
    nodes = pd.read_parquet(PROCESSED / name / "nodes.parquet")
    edges = pd.read_parquet(PROCESSED / name / "edges.parquet")
    ranks = pd.read_parquet(RESULTS / name / "ranks.parquet")
    info = json.loads((RESULTS / name / "run_info.json").read_text())
    df = ranks.merge(nodes.drop(columns=[c for c in ("cats",) if c in nodes.columns]), on="id", how="left")
    df["label"] = df[LABEL[name]].astype(str)
    df["in_degree"] = df.id.map(edges.dst.value_counts()).fillna(0).astype(int)
    df["out_degree"] = df.id.map(edges.src.value_counts()).fillna(0).astype(int)
    return df, edges, info

data = {n: load(n) for n in DATASETS}
resumen = pd.DataFrame({
    n: {"nodos": info["nodos"], "aristas": info["aristas"], "sin_salida": info["nodos_sin_salida"],
        "iter PageRank": info["pagerank"]["iteraciones"], "iter HITS": info["hits"]["iteraciones"],
        "iter TSPR": info["tspr"]["iteraciones"], "segundos": info["segundos_total"]}
    for n, (_, _, info) in data.items()}).T
resumen.to_csv(TABLES / "resumen_datasets.csv")
resumen

## 2. Distribución de grados (log-log)

In [ ]:
fig, axes = plt.subplots(1, len(data), figsize=(4.5 * len(data), 3.6), squeeze=False)
for ax, (n, (df, _, _)) in zip(axes[0], data.items()):
    for col, lab in (("in_degree", "entrada"), ("out_degree", "salida")):
        vc = df[col].value_counts().sort_index()
        vc = vc[vc.index > 0]
        ax.loglog(vc.index, vc.values, ".", ms=4, label=lab)
    ax.set_title(n); ax.set_xlabel("grado"); ax.set_ylabel("nodos"); ax.legend()
fig.tight_layout(); fig.savefig(FIGS / "distribucion_grados.pdf"); plt.show()

## 3. Top-20 por algoritmo y dataset

In [ ]:
def top(df, col, k=20):
    t = df.sort_values(col, ascending=False).head(k)[["label", col, "in_degree", "out_degree"]]
    return t.reset_index(drop=True)

for n, (df, _, _) in data.items():
    for col in ("pagerank", "authority", "hub"):
        t = top(df, col)
        t.to_csv(TABLES / f"top20_{n}_{col}.csv", index=False)
        print(f"\n=== {n}: top-20 {col}")
        display(t)

## 4. PageRank vs HITS vs grado de entrada

In [ ]:
filas = []
for n, (df, _, _) in data.items():
    s = df[["pagerank", "authority", "hub", "in_degree", "out_degree"]].rank()
    corr = s.corr()
    filas.append({"dataset": n,
                  "PR~in_degree": corr.loc["pagerank", "in_degree"],
                  "PR~authority": corr.loc["pagerank", "authority"],
                  "PR~hub": corr.loc["pagerank", "hub"],
                  "authority~hub": corr.loc["authority", "hub"],
                  "top20 PR∩authority": len(set(top(df, "pagerank").label) & set(top(df, "authority").label)) / 20,
                  "top20 PR∩in_degree": len(set(top(df, "pagerank").label) & set(top(df, "in_degree").label)) / 20})
cmp_df = pd.DataFrame(filas).set_index("dataset").round(3)
cmp_df.to_csv(TABLES / "comparacion_algoritmos.csv")
cmp_df   # Spearman entre rangos y solapamiento del top-20

**Nota para Spotify:** la red es no dirigida (cada colaboración son dos aristas), así que `authority` y `hub` coinciden
y PageRank es casi proporcional al grado. Es un efecto del grafo simétrico, no un error del algoritmo.

## 5. Convergencia

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, (key, titulo) in zip(axes, (("pagerank", "PageRank"), ("hits", "HITS"), ("tspr", "PageRank sensible a tópico"))):
    for n, (_, _, info) in data.items():
        ax.semilogy(range(1, len(info[key]["historia"]) + 1), info[key]["historia"], label=n)
    ax.set_title(titulo); ax.set_xlabel("iteración"); ax.set_ylabel("distancia L1"); ax.legend()
fig.tight_layout(); fig.savefig(FIGS / "convergencia.pdf"); plt.show()

## 6. PageRank sensible a tópico: top-10 por tópico

In [ ]:
def topic_cols(df): return [c for c in df.columns if c.startswith("tspr_")]

for n, (df, _, _) in data.items():
    cols = topic_cols(df)
    tabla = pd.DataFrame({c.removeprefix("tspr_"): df.sort_values(c, ascending=False).head(10).label.values for c in cols})
    tabla.index = range(1, 11)
    tabla.to_csv(TABLES / f"tspr_top10_{n}.csv")
    print(f"\n=== {n}: top-10 por tópico")
    display(tabla)

### Combinación en el momento de la consulta
`r = Σ_t w_t · r_t`: los pesos `w_t` expresan el interés del usuario (p. ej. 70 % ciencia + 30 % tecnología).
Cambia `PESOS` por tópicos que existan en cada dataset (ver columnas `tspr_*`).

In [ ]:
PESOS = {
    "hepph": {"1992_1994": 0.7, "1995_1997": 0.3},
    "spotify": {"rock": 0.6, "electronica": 0.4},
    "wiki_sample": {"ciencia": 0.7, "tecnologia": 0.3},
    "wiki_full": {"ciencia": 0.7, "tecnologia": 0.3},
}
for n, (df, _, _) in data.items():
    w = {("tspr_" + t): v for t, v in PESOS.get(n, {}).items() if ("tspr_" + t) in df.columns}
    if not w:
        print(n, ": ningún tópico de PESOS existe en este dataset"); continue
    total = sum(w.values())
    q = sum(df[c] * v / total for c, v in w.items())
    resultado = df.assign(consulta=q).sort_values("consulta", ascending=False).head(10)[["label", "consulta", "pagerank"]]
    print(f"\n=== {n}: consulta {PESOS[n]}")
    display(resultado.reset_index(drop=True))

## 7. Qué anotar en el informe
- Tamaños antes y después de la poda de Wikipedia (`data/processed/wiki_*/graph_stats.json`).
- Diferencia entre la muestra y el dump completo (si terminó): solapamiento de los top-20.
- Validación contra networkx (`src/baselines/reference_networkx.py`): errores máximos.
- Limitaciones: solo enlaces del wikitext; la muestra es un prefijo del dump; tópicos por palabras clave.